# Telegram Expense Tracker — Demo evaluada

Notebook principal de la tarea final. Se ejecuta de principio a fin y en orden, sin Telegram.
Cada sección corresponde a un criterio de la rúbrica o a un bono declarado (ver `docs/bonos.md`).

| Sección | Contenido | Estado |
|---|---|---|
| 0 | Setup y ficha de reproducción | **Disponible** |
| 1 | Caso de uso y criterio de éxito | **Disponible** |
| 2 | LLM y prompts | **Disponible** (llamadas reales: requieren `.env`) |
| 3 | ReAct | Pendiente |
| 4 | Historial | Pendiente |
| 5 | Seguridad | Pendiente |
| 6 | Router (bono) | Pendiente |
| 7 | Memoria avanzada (bono) | Pendiente |
| 8 | Herramienta de acción (bono) | Pendiente |
| 9 | Juez LLM (bono) | Pendiente |
| 10 | Golden set (bono) | Pendiente |

## Sección 0 — Setup y ficha de reproducción

Esta sección prepara el entorno y no requiere `.env` ni red: solo muestra si cada variable está **definida** o **falta** (nunca su valor) y demuestra el trazador con un secreto falso que se enmascara. El modelo exacto se completa en la Etapa 3.

In [ ]:
import sys
from pathlib import Path

# El notebook vive en notebooks/: se agrega la raíz del repositorio al path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.version.split()[0])
print("Raíz del repositorio:", ROOT.name)

In [ ]:
from app.config import config_status

# Solo presencia de variables, nunca valores.
for name, state in config_status().items():
    print(f"{name:<34} {state}")

### Demostración del trazador

Se registra un evento con un secreto **falso** construido en tiempo de ejecución. La consola y el archivo JSONL solo muestran `***`.

In [ ]:
import tempfile

from app.trace import Tracer

fake_key = "AIza" + "x" * 35  # secreto de ejemplo, no real
with tempfile.TemporaryDirectory() as tmp:
    tracer = Tracer(session="demo", trace_dir=tmp)
    tracer.record("USER_INPUT", {"texto": "Registra este recibo"})
    tracer.record("TOOL_CALL", {"tool": "analizar_recibo", "api_key": fake_key,
                                "nota": f"clave usada: {fake_key}"})
    contenido = tracer.path.read_text(encoding="utf-8")

print("Eventos:", len(tracer.events))
print("El secreto aparece en el JSONL:", fake_key in contenido)

## Sección 1 — Caso de uso y criterio de éxito

**Usuario:** persona que registra sus gastos personales y usa Telegram a diario.
**Entrada:** imagen de un recibo (JPG/PNG) y texto opcional.
**Salida:** archivo en Drive, fila nueva en Sheets y confirmación con el número de fila.

**Criterio observable:** dada una imagen válida de `data/receipts/`, el caso es exitoso cuando
(1) fecha, comercio y monto coinciden con `data/receipts/expected.json`;
(2) la categoría es una de las permitidas;
(3) Drive devuelve un `file_id` y un `web_view_link` que existen;
(4) la planilla tiene exactamente una fila más, con ese enlace;
(5) la respuesta contiene el número de fila.

**Por qué un LLM:** los recibos no tienen estructura fija y el flujo exige decidir entre registrar, pedir confirmación, pedir otra foto o rechazar.
**Por qué herramientas:** el LLM no puede persistir; Drive y Sheets devuelven observaciones verificables.

La celda siguiente muestra el documento completo del caso de uso.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
display(Markdown((ROOT / "docs" / "use_case.md").read_text(encoding="utf-8")))

## Sección 2 — LLM y prompts

Criterio de rúbrica: **LLM real y trazabilidad** (modelo, configuración, prompts y traza de cada llamada).

- Modelo de ejecución confirmado por el autor: `gemini-3.5-flash-lite` (SDK oficial `google-genai`). El código **no** lo escribe: lo lee de `LLM_MODEL`.
- Todas las llamadas incluyen `SECURITY_SCOPE_v1`; el texto de las imágenes es **dato, no instrucción**.
- La última celda llama al LLM solo si hay `GEMINI_API_KEY` y `LLM_MODEL`; sin ellas se omite con un aviso y el notebook sigue corriendo.

In [ ]:
from app.config import config_status
from app.llm import EXTRACTION_TEMPERATURE

status = config_status()  # solo presencia, nunca valores
print(f"{'Variable':<34} Estado")
for name in ("GEMINI_API_KEY", "LLM_MODEL", "LLM_MAX_RETRIES", "LLM_MIN_SECONDS_BETWEEN_CALLS"):
    print(f"{name:<34} {status[name]}")
print()
print("ID de modelo confirmado por el autor: gemini-3.5-flash-lite (LLM_MODEL debe tenerlo)")
print("Fuentes: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite")
print("         https://ai.google.dev/gemini-api/docs/pricing")
print(f"Temperatura de extracción (EXTRACTION_TEMPERATURE): {EXTRACTION_TEMPERATURE}")
print("  0.0 según el prompt maestro; pendiente de verificación real")
print("  (Google recomienda 1.0 en Gemini 3: https://ai.google.dev/gemini-api/docs/gemini-3)")

### Prompts versionados

Todos los prompts viven en `app/prompts.py`. La traza registra el identificador, no el texto completo.

In [ ]:
from app.prompts import PROMPTS, compose_system_instruction

print("Identificadores de prompt:", ", ".join(PROMPTS))
print()
print(compose_system_instruction("ANALYZER_PROMPT_v1"))

### Recibos sintéticos

Imágenes generadas con `scripts/generate_receipts.py` (comercios ficticios, sin datos personales). Ver `data/README.md`.

In [ ]:
import json
from IPython.display import Image, display

RECEIPTS = ROOT / "data" / "receipts"
expected = json.loads((RECEIPTS / "expected.json").read_text(encoding="utf-8"))
for name, exp in expected.items():
    print(f"{name}: esperado {exp}")
    display(Image(filename=str(RECEIPTS / name), width=360))

### Llamada real a `analizar_recibo`

Se ejecuta solo si `GEMINI_API_KEY` y `LLM_MODEL` están definidas. Cada recibo consume una llamada de la capa gratuita; la traza muestra el modelo, los parámetros, los tokens y los reintentos ante 429.

In [ ]:
from app.llm import LLMCallError, LLMClient
from app.tools.analyzer import analizar_recibo
from app.trace import Tracer

if status["GEMINI_API_KEY"] == "definida" and status["LLM_MODEL"] == "definida":
    tracer = Tracer(session="notebook-seccion-2")
    llm = LLMClient(tracer=tracer)
    for name, exp in expected.items():
        print(f"\n== {name} (esperado: fecha={exp['fecha']}, monto={exp['monto']}) ==")
        try:
            print("Resultado:", analizar_recibo(RECEIPTS / name, llm=llm).model_dump())
        except LLMCallError as error:
            print("La API falló tras los reintentos:", error)
    print("\nContadores de la sesión:", llm.stats.as_dict())
else:
    print("omitido: falta GEMINI_API_KEY/LLM_MODEL (configura .env; ver .env.example).")
    print("Las secciones anteriores y las pruebas offline no los necesitan.")